In [ ]:
import re
import matplotlib.pyplot as plt
from pathlib import Path

log_dir = Path("logs")
log_files = sorted(log_dir.glob("lr*.txt"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for log_file in log_files:
    lr_label = log_file.stem 
    text = log_file.read_text()

    # Parse training loss
    train_matches = re.findall(
        r"iter\s+(\d+)/\d+\s+\|\s+loss\s+([\d.]+)\s+\|", text
    )
    train_iters = [int(m[0]) for m in train_matches]
    train_losses = [float(m[1]) for m in train_matches]

    # Parse validation loss
    val_matches = re.findall(
        r"iter\s+(\d+)/\d+\s+\|\s+val_loss\s+([\d.]+)", text
    )
    val_iters = [int(m[0]) for m in val_matches]
    val_losses = [float(m[1]) for m in val_matches]

    axes[0].plot(train_iters, train_losses, label=lr_label)
    axes[1].plot(val_iters, val_losses, label=lr_label, marker="o")

axes[0].set_title("Training Loss vs. Iteration")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("Validation Loss vs. Iteration")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Val Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/lr_loss.png")
plt.show()

In [ ]:
log_files = {
    "1": log_dir / "batch_1.txt",
    "64": log_dir / "batch_64.txt",
    "128": log_dir / "lr1e-3.txt",   # batch 128 run
    "256": log_dir / "batch_256.txt",
    "512": log_dir/ "batch_512.txt",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

MAX_STEP = 20000

for batch_size, log_file in log_files.items():
    text = log_file.read_text()

    # training loss
    train_matches = re.findall(
        r"iter\s+(\d+)/\d+\s+\|\s+loss\s+([\d.]+)\s+\|", text
    )
    train_iters = [int(m[0]) for m in train_matches]
    train_losses = [float(m[1]) for m in train_matches]

    # validation loss
    val_matches = re.findall(
        r"iter\s+(\d+)/\d+\s+\|\s+val_loss\s+([\d.]+)", text
    )
    val_iters = [int(m[0]) for m in val_matches]
    val_losses = [float(m[1]) for m in val_matches]

    # filter to <= 50k
    train_data = [(i, l) for i, l in zip(train_iters, train_losses) if i <= MAX_STEP]
    val_data = [(i, l) for i, l in zip(val_iters, val_losses) if i <= MAX_STEP]

    if train_data:
        ti, tl = zip(*train_data)
        axes[0].plot(ti, tl, label=f"batch={batch_size}")

    if val_data:
        vi, vl = zip(*val_data)
        axes[1].plot(vi, vl, marker="o", label=f"batch={batch_size}")

axes[0].set_title("Training Loss vs Iteration (≤20k)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("Validation Loss vs Iteration (≤20k)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Validation Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/batch_size_loss_20k.png")
plt.show()

In [ ]:
owt_log_text = (log_dir / "owt_train_result.txt").read_text()
train_re = re.compile(
    r"iter (\d+)/\d+ \| loss ([0-9.]+) \| lr ([0-9.eE+-]+) \| elapsed ([0-9.]+)s"
)
val_re = re.compile(r"iter (\d+)/\d+ \| val_loss ([0-9.]+)")

train_steps = []
train_losses = []
train_elapsed_map = {}

for line in owt_log_text.splitlines():
    m = train_re.search(line)
    if m:
        step = int(m.group(1))
        loss = float(m.group(2))
        elapsed = float(m.group(4))
        train_steps.append(step)
        train_losses.append(loss)
        train_elapsed_map[step] = elapsed

val_steps = []
val_losses = []
val_elapsed = []

for line in owt_log_text.splitlines():
    m = val_re.search(line)
    if m:
        step = int(m.group(1))
        val_loss = float(m.group(2))
        if step in train_elapsed_map:
            val_steps.append(step)
            val_losses.append(val_loss)
            val_elapsed.append(train_elapsed_map[step] / 60.0)  # minutes

plt.figure(figsize=(5, 3.5))
plt.plot(train_steps, train_losses, marker="o", label="Training loss")
plt.plot(val_steps, val_losses, marker="s", label="Validation loss")
plt.xlabel("Training steps")
plt.ylabel("Loss")
plt.title("Training/validation loss vs steps")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("plots/owt_loss.png", dpi=200)
plt.show()

print("Final val_loss:", val_losses[-1])
print("Total wallclock (min):", val_elapsed[-1])